# Workflow

To process a document, we intend to the following things:
1. Ingest the data from the file.
2. Break the data into chunks that easily fit into the context window of the model.
3. Create vector embedding for the vector database to retrieve it.
4. Put it to a vector DB.

# Data Ingestion

In [ ]:
import os                                                               # For directory and file creation

In [ ]:
# Libraries that are required for document loading
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader             # For loading tect
from langchain_community.document_loaders import DirectoryLoader        # For loading directories
from langchain_community.document_loaders import PyMuPDFLoader          # For loading PDFs
from langchain_community.document_loaders import PyPDFLoader            # For loading PDFs
from langchain_text_splitters import RecursiveCharacterTextSplitter     # For chunking


/home/rahu_g/.anaconda3/envs/LLM/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Libraries required for vector store
import numpy as np                                                      # For storing embeddings as ndarray
import sklearn                                                          # Required by sentence_transformers
import chromadb                                                         # The vector store
import uuid                                                             # For generating unique document IDs
from typing import List, Dict, Any, Tuple                               # For data storing and manipulation. These are non-primitive data types (object of List, Dict, Any and Tuple). Required as loaders return non-primitive datatypes.
from sklearn.metrics.pairwise import cosine_similarity                  # For calculating similarity between query and stored document embeddings
from sentence_transformers import SentenceTransformer                   # Embeddings will be generated from transformers provided by this library

E0000 00:00:1776182461.285643 3692142 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1776182461.285668 3692142 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1776182461.285669 3692142 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1776182461.285671 3692142 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1776182461.285672 3692142 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


### Langchain document

Langchin document consists of the page_content and the metadata. The page_content will store the actual text, and the metadata will store the supporting information, like author, page number, creation date, last edit date, etc. 

In [4]:
doc = Document(
    page_content="This is the content of the document.",
    metadata={"source": "example.txt",
              "page": 1,
              "author": "John Doe",
              "date_created": "2024-06-01"},
)
print(doc)
print(f"Type of doc: {type(doc)}")

page_content='This is the content of the document.' metadata={'source': 'example.txt', 'page': 1, 'author': 'John Doe', 'date_created': '2024-06-01'}
Type of doc: <class 'langchain_core.documents.base.Document'>


## 1. Create the directories and files

In [5]:
# os.mkdir("./data")
# os.mkdir("./data/text_files")
# os.mkdir("./data/pdf_files")

Here created four documents to put them on a file.

In [6]:
sample_doc = {"./data/text_files/python.txt": """## Python Programming: A Comprehensive Overview

Python is one of the most popular, versatile, and beginner-friendly programming languages in the world. It was created by Guido van Rossum and emphasizes code readability, allowing programmers to express complex concepts in fewer 
lines of code than languages like C++ or Java.

In simple terms, Python provides a powerful, yet accessible, way to write software using a syntax that closely resembles plain English.

***

### Part 1: What Makes Python Unique? (The Philosophy)

The core philosophy behind Python is **simplicity and readability.**

#### 1. High-Level Language
Python is classified as a high-level language. This means that it is designed to be close to human language and abstracts away complex, low-level details (such as managing computer memory or dealing with machine registers). 
Programmers can therefore focus on *what* the program needs to achieve, rather than *how* the computer must execute every single step.

#### 2. Interpreted Execution
When you write code in Python, it is typically executed by an **interpreter**.

*   **How it works:** The interpreter reads your code line by line and executes it immediately.
*   **The Advantage:** This makes Python incredibly fast to test, debug, and iterate on, which significantly boosts productivity for beginners and rapid development teams. (Contrast this with compiled languages, where the entire 
program must be translated into machine code before it can run.)

#### 3. Dynamic and Strongly Typed
*   **Dynamic:** You do not need to explicitly declare the data type of a variable when you create it. Python determines the type as the code runs.
    (Example: If you write `x = 5`, Python recognizes `x` as a number. Later, if you write `x = "hello"`, Python allows this type change.)
*   **Strongly:** While dynamic, Python is strongly typed, meaning once a variable holds a certain type of data, it won't let you accidentally treat it as a fundamentally different type without proper conversion.

***

### Part 2: Why is Python So Popular? (The Ecosystem)

Python’s success is due not only to its syntax but also to its massive standard library and community support structure.

#### Versatility (The "Swiss Army Knife")
Python is unmatched in its versatility. It can be used across nearly every field of technology. This minimizes the need for a developer to learn dozens of disparate tools.

#### Vast Ecosystem of Libraries
The true power of Python comes from its third-party packages and frameworks, which are pre-written, tested blocks of code that solve common problems.

*   **Package Management:** The package installer, `pip`, grants access to hundreds of thousands of specialized, reliable libraries.
*   **Example:** If you need to process images, you do not write the code from scratch; you simply import a powerful library like Pillow and utilize its pre-written functions.

***

### Part 3: Where is Python Used? (Major Applications)

Python is a dominant force across several major industry sectors:

#### 1. Data Science and Machine Learning
This is currently one of Python's most significant fields. It is the industry standard for data analysis, scientific computing, and AI development.
*   **Key Libraries:**
    *   **Pandas:** Used for robust data manipulation and analysis (handling structured data tables).
    *   **NumPy:** Used for complex, high-performance mathematical operations on large numerical arrays.
    *   **Matplotlib / Seaborn:** Used for creating sophisticated data visualizations (charts and graphs).
    *   **Scikit-learn / TensorFlow / PyTorch:** Industry-leading tools for building complex machine learning models and Artificial Intelligence systems.

#### 2. Web Development (Backend)
Python excels at building the "backend" (the server side) of websites and web applications—the crucial component that handles the logic, database interaction, and user authentication.
*   **Key Frameworks:**
    *   **Django:** A powerful, "batteries-included" framework perfect for large, complex, and structured sites (e.g., full social networking platforms).
    *   **Flask:** A lightweight framework, ideal for building smaller, highly focused services or APIs (microservices).

#### 3. Automation and Scripting
Python is excellent for automating repetitive, tedious tasks that used to require manual human effort.
*   **Practical Examples:**
    *   Renaming thousands of files automatically.
    *   Scraping data from specific websites (web scraping).
    *   Generating and sending structured reports via email on a set schedule.
    *   Interacting with the operating system's files and processes.

#### 4. Gaming and Tooling
While not typically used for ultra-high-fidelity 3D graphics (which usually require specialized engines), Python is invaluable for prototyping game logic, creating simple games, and building internal developer tools.

***

### Pros and Cons (A Balanced View)

| Advantages (Pros) | Disadvantages (Cons) |
| :--- | :--- |
| **Learning Curve:** Simple syntax makes it exceptionally ideal for absolute beginners. | **Execution Speed:** Python can be slower than low-level languages like C++ or Rust when required to perform massive, real-time computational 
tasks. |
| **Community Support:** Due to its age and popularity, any problem encountered has likely already been solved and documented online. | **Complexity Management:** For high-performance, multi-threaded concurrency, advanced concepts 
like the Global Interpreter Lock (GIL) must be understood. |
| **Readability:** Code is clean and resembles pseudocode, making large projects easier to maintain and debug for new developers. | **Library Bloat:** The massive ecosystem means that new developers can sometimes get overwhelmed by 
the sheer volume of available tools and dependencies. |
| **Versatility:** One language that can transition you between data analysis, web development, and system scripting. | |

***

### How to Get Started

If you are interested in learning Python, remember that **the best way to learn is by building things.**

1.  **Master the Fundamentals:** Focus initially on variables, data types (strings, integers, lists), control flow (loops like `for` and `while`), conditional logic (`if/else`), and defining functions.
2.  **Build Small Projects:** Do not attempt to build Google immediately. Start with small, contained projects:
    *   A simple command-line calculator.
    *   A script that checks for files in a local directory.
    *   A "Guess the Number" game.
3.  **Specialization:** After the fundamentals, choose a specific area that interests you (Data Science, Web Dev, or Automation) and focus your learning on the core libraries and frameworks for that domain (e.g., Django for web, 
Pandas for data).
""",
"./data/text_files/java.txt": """## Java Programming: A Comprehensive Overview

Java is a highly established, object-oriented programming language designed for building large-scale, robust, and scalable applications. Developed initially by Sun Microsystems and now primarily maintained by Oracle, Java has been a 
cornerstone of enterprise software development for decades.

Its design philosophy centers on reliability and portability, encapsulated by the famous motto: "Write Once, Run Anywhere."

***

### I. Core Philosophy and Pillars

The enduring strength of Java lies in the strict adherence to fundamental computer science principles.

#### 1. Object-Oriented Programming (OOP)
Java is a purely object-oriented language. This paradigm dictates that programs should be structured around "objects," which are instances of "classes." OOP mandates four core concepts:

*   **Encapsulation:** Bundling data (attributes) and the methods (behavior) that operate on that data into a single unit (the class). This hides internal complexity and protects data integrity.
*   **Inheritance:** Allowing a new class (child) to adopt the properties and behaviors of an existing class (parent). This promotes code reusability.
*   **Polymorphism:** The ability for an object or method to take on many forms. This means one interface can be used to represent different underlying data types.
*   **Abstraction:** Showing only the essential information to the user and hiding the complex implementation details.

#### 2. Platform Independence
This is Java's most notable feature. Java code is compiled into **bytecode**, not native machine code. This bytecode is then executed by the Java Virtual Machine (JVM). Because the JVM is available for various operating systems 
(Windows, macOS, Linux), the same compiled Java bytecode can run on any platform without modification.

#### 3. Robustness and Memory Management
Java is designed for reliability.
*   **Strong Typing:** Java is strongly typed, meaning the programmer must explicitly declare the type of every variable. This catches many programming errors during compilation, rather than allowing them to fail at runtime.
*   **Automatic Garbage Collection (GC):** Java manages memory automatically. Programmers do not have to manually allocate and deallocate memory resources (a common source of bugs in languages like C++), making the development 
process safer and more robust.

---

### II. Ecosystem and Use Cases

Java's robust, mature ecosystem has allowed it to become the backbone of mission-critical systems.

*   **Enterprise Backend Systems:** Java is the dominant language for large, scalable enterprise applications (e.g., banking, insurance, e-commerce backends) due to its stability and performance under heavy load.
*   **Android Mobile Development:** Historically and significantly, Java remains a core language for native Android application development (though Kotlin is increasingly favored).
*   **Big Data Technologies:** Many leading big data frameworks (like Hadoop and Apache Kafka) are built using Java, solidifying its role in data processing pipelines.
*   **Web Applications:** Frameworks like Spring Boot have made building RESTful APIs and microservices on the Java stack extremely efficient.

---

### III. Pros and Cons

| Aspect | Pros (Strengths) | Cons (Weaknesses) |
| :--- | :--- | :--- |
| **Stability** | Exceptionally stable, highly reliable, and predictable performance under load. | Verbose (requires more boilerplate code than modern languages). |
| **Ecosystem** | Massive, mature ecosystem with extensive libraries and community support. | Can be overly complex for simple projects due to the sheer size of the framework options. |
| **Safety** | Strong type checking and automatic memory management (Garbage Collection) prevent many common bugs. | Historically known for long compile times (though modern IDEs have improved this significantly). |
| **Performance**| Excellent performance due to the sophisticated Just-In-Time (JIT) compiler used by the JVM. | Can have a steep learning curve, especially for beginners. |

---

### Summary Takeaway

Java is a **workhorse language**. It may not be the fastest language to write simple code in, but it is one of the most reliable, scalable, and battle-tested languages available for building complex, large-scale, enterprise-grade 
systems that must run correctly 24/7. Its stability and robust ecosystem are its greatest strengths.""",

"./data/text_files/cpp.txt" : """## Detailed Overview of C Programming

C is a general-purpose, procedural computer programming language developed by Dennis 
Ritchie at Bell Labs between 1972 and 1973. It is renowned for its efficiency, 
portability, and low-level memory access capabilities, making it foundational to 
modern computing. Due to its nature, it is often used to write operating systems, 
embedded systems, and compilers.

### I. Core Concepts and Philosophy

**1. Procedural Paradigm:**
C is primarily a procedural language. This means that the program structure revolves 
around a sequence of steps or routines (procedures/functions) that operate on data. Unlike object-oriented languages, C does not inherently enforce concepts like inheritance or polymorphism; structure and organization are achieved 
through functions and data structures.

**2. System-Level Programming:**
One of C's defining features is its proximity to hardware. It allows programmers to manipulate memory directly using pointers. This level of control is why C is often the language of choice for developing kernel code (e.g., Linux, 
Windows components) and firmware.

**3. Portability:**
While C code must be compiled for a specific architecture (e.g., x86, ARM), the language itself is highly portable. With adherence to standards (like ANSI C or C99/C11), a well-written C program can generally be compiled and run on 
different types of hardware with minimal modifications.

### II. Key Features and Mechanisms

**1. Pointers:**
Pointers are the most critical and unique feature of C. A pointer is a variable that stores the memory address of another variable. They enable efficient array manipulation, function parameter passing (by reference), and direct 
memory access. Mastering pointers is essential to mastering C.

**2. Memory Management:**
C provides explicit manual memory management. Programmers use the standard library functions `malloc()` (memory allocation) and `calloc()` (memory allocation and initialization) to request blocks of memory from the heap. Crucially, 
they must use `free()` to return this memory to the system to prevent memory leaks. This manual control is powerful but demanding, as improper use can lead to segmentation faults and system instability.

**3. Data Types:**
C supports a limited but powerful set of data types:
*   **Basic Types:** `int` (integers), `char` (single characters), `float` (single-precision floating point), `double` (double-precision floating point).
*   **Derived Types:** Arrays (fixed-size collections of homogeneous elements) and Pointers.
*   **User-Defined Types:** `struct` (structures), which allow grouping variables of different types under a single name. `union` (unions), which allow multiple members to occupy the same memory space.

**4. Preprocessor Directives:**
The C preprocessor is a program that runs before the actual compilation phase. It handles directives starting with `#`, such as:
*   `#include`: Inserts the content of header files (e.g., `<stdio.h>`).
*   `#define`: Defines preprocessor constants or simple macros (text substitution).

### III. Compilation and Execution Flow

The development process using C generally follows these steps:

1.  **Writing Code:** Writing source code in a file with a `.c` extension.
2.  **Preprocessing:** The preprocessor handles directives (`#include`, `#define`), expanding macros and incorporating header file contents.
3.  **Compilation:** The compiler takes the preprocessed code and translates it into assembly language and then into machine-specific assembly code.
4.  **Assembly:** The assembler converts the assembly code into machine object files (`.o`).
5.  **Linking:** The linker combines the object files and any required external library routines (like standard I/O functions) to create the final, executable program.

### IV. Advantages and Disadvantages

**Advantages:**
*   **Speed and Efficiency:** Because it compiles directly to machine code and allows direct memory manipulation, C programs run extremely fast with minimal runtime overhead.
*   **Resource Control:** Ideal for developing software with strict memory or computational resource constraints (embedded systems).
*   **Foundation:** It is the base language for many other high-level languages (including C++, Python, and shell scripting tools).

**Disadvantages:**
*   **Complexity and Difficulty:** Manual memory management and pointer arithmetic make the language difficult for beginners and prone to subtle, hard-to-debug errors (e.g., buffer overflows, dangling pointers).
*   **Lack of Abstraction:** The programmer must manage many low-level details that higher-level languages handle automatically (e.g., garbage collection, boundary checks).
*   **Security Risks:** The direct access to memory can make C programs susceptible to security vulnerabilities if coding practices are not rigorously followed.
""",

"./data/text_files/cuda.txt" : """## Detailed Overview of CUDA Programming

CUDA (Compute Unified Device Architecture) is a parallel computing platform and programming model developed by NVIDIA. It allows developers to utilize the massive parallel processing capabilities of NVIDIA Graphics Processing Units 
(GPUs) for general-purpose computation, moving beyond traditional graphics rendering tasks. Essentially, CUDA enables the use of the GPU as a highly powerful, specialized co-processor for complex scientific, machine learning, and 
data processing algorithms.

### I. Foundational Concepts

**1. The Need for CUDA:**
Traditional CPU (Central Processing Unit) architectures are optimized for sequential processing—performing tasks step-by-step using a few, very powerful cores. However, many modern computational problems, such as matrix 
multiplications, physics simulations, and deep learning inference, are "embarrassingly parallel," meaning the same operation needs to be performed simultaneously on thousands or millions of independent data points. GPUs, with their 
architecture of hundreds or thousands of smaller, highly parallel cores, are perfectly suited for this type of work.

**2. GPU Architecture (SIMT):**
GPUs operate under the **Single Instruction, Multiple Thread (SIMT)** model. This means that a single instruction is issued, and multiple threads execute that instruction simultaneously across different data points. This contrasts 
with the CPU's primary focus on deep pipeline optimization for sequential tasks.

**3. CUDA Scope:**
It is critical to understand that CUDA is a platform, not a language itself. It is an API and toolkit that allows programmers to use extensions to standard languages, primarily C and C++, to manage data transfer, kernel execution, 
and memory communication between the CPU and the GPU.

### II. The CUDA Programming Model

The core of CUDA programming involves restructuring a sequential algorithm to run in a massively parallel fashion. This process relies on defining and launching special functions called **kernels**.

**1. Kernel Functions:**
A kernel is a function written in CUDA C/C++ that is designed to execute *on the GPU*. When the CPU initiates the kernel, it is not executing the code itself; rather, it is configuring the GPU's streaming multiprocessors to execute 
the kernel simultaneously across many threads.

**2. Parallel Hierarchy:**
CUDA organizes the execution environment into a strict, hierarchical structure:
*   **Grid:** The entire set of concurrently running threads. It represents the total scope of the problem (e.g., processing a massive array).
*   **Block:** A group of threads within the grid that can communicate quickly with each other using dedicated on-chip memory (Shared Memory). Threads within the same block can synchronize and cooperate, which is essential for 
complex algorithms like reductions.
*   **Thread:** The smallest unit of execution. Each thread performs a specific, independent calculation instance.

**3. Memory Space:**
The performance of a CUDA program is heavily dependent on understanding the memory hierarchy:
*   **Global Memory:** The largest, highest-capacity memory (VRAM) on the GPU. It is accessible by all threads but involves the highest latency (slowest access).
*   **Shared Memory:** Small, extremely fast, on-chip memory local to a single block. It is used by cooperating threads within the block to pass intermediate results, dramatically speeding up computation.
*   **Registers:** The fastest memory, local to a single thread. This is where the thread's immediate variables reside.

### III. Execution Flow and Computation Steps

A typical CUDA program follows these functional steps:

1.  **Setup (CPU Side):** The CPU code initializes variables and allocates necessary data structures.
2.  **Data Transfer (Host to Device):** The initial input data (on the CPU's RAM, called *Host* memory) must be explicitly copied to the GPU's dedicated memory (called *Device* memory). This transfer is a major performance 
bottleneck and must be managed efficiently.
3.  **Kernel Launch (CPU to GPU):** The CPU launches the kernel, providing launch configuration parameters: the dimensions of the grid and the dimensions of the blocks.
4.  **Execution (GPU Side):** The GPU's Streaming Multiprocessors simultaneously execute the kernel function across all configured threads, performing the parallel computation.
5.  **Result Transfer (Device to Host):** Once the kernel completes, the computed results (on Device memory) must be explicitly copied back to the CPU's Host memory for further processing or reporting.

### IV. Advantages and Use Cases

**Advantages:**
*   **Scalability:** Offers unparalleled computational power for massively parallel tasks.
*   **Speedup:** Provides exponential speedups over CPU-only implementations for suitable algorithms.
*   **Versatility:** Applicable across various domains, from AI to engineering.

**Common Use Cases:**
*   **Deep Learning:** Training and running neural networks (e.g., TensorFlow, PyTorch).
*   **Scientific Simulation:** Computational fluid dynamics (CFD), molecular dynamics.
*   **Data Analysis:** Large-scale matrix operations and graph processing.
*   **Image and Video Processing:** Filtering, transformation, and analysis.

**Challenges and Considerations:**
*   **Data Transfer Overhead:** Excessive data movement between Host and Device memory can negate the benefits of parallel processing. Algorithms must be structured to keep data on the GPU for as long as possible.
*   **Complexity:** The memory hierarchy, explicit memory management, and parallel structure significantly increase the complexity of the programming model compared to standard CPU programming.
*   **Hardware Dependency:** CUDA is inherently tied to NVIDIA hardware and requires specific hardware for execution."""

}

Writing the sample documents to files

In [7]:
for filename, content in sample_doc.items():
    with open(filename, "w") as f:
        f.write(content)

## 2. Retrieve the text documents using Langchain TextLoader

Loading the text file using TextLoader, as we are trying to load simple text file with utf-8 encoding

In [8]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./data/text_files/python.txt",encoding="utf8")                 # The TextLoader object should be initialised with the file path and encoding format
print(loader)
print(f"Type of loader: {type(loader)}")

Type of loader: <class 'langchain_community.document_loaders.text.TextLoader'>


In [9]:
document = loader.load()                                                            # The member function load() loads the file from the file path and 
print(document)
print(f"Document type: {type(document)}")

[Document(metadata={'source': './data/text_files/python.txt'}, page_content='## Python Programming: A Comprehensive Overview\n\nPython is one of the most popular, versatile, and beginner-friendly programming languages in the world. It was created by Guido van Rossum and emphasizes code readability, allowing programmers to express complex concepts in fewer \nlines of code than languages like C++ or Java.\n\nIn simple terms, Python provides a powerful, yet accessible, way to write software using a syntax that closely resembles plain English.\n\n***\n\n### Part 1: What Makes Python Unique? (The Philosophy)\n\nThe core philosophy behind Python is **simplicity and readability.**\n\n#### 1. High-Level Language\nPython is classified as a high-level language. This means that it is designed to be close to human language and abstracts away complex, low-level details (such as managing computer memory or dealing with machine registers). \nProgrammers can therefore focus on *what* the program needs

In [10]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(   "./data/text_files", 
                                glob="**/*.txt",
                                loader_cls=TextLoader,
                                loader_kwargs={'encoding': 'utf8'},
                                show_progress=True
                            )
print(dir_loader)
print(f"Type of dir_loader: {type(dir_loader)}")

Type of dir_loader: <class 'langchain_community.document_loaders.directory.DirectoryLoader'>


In [11]:
documents = dir_loader.load()
print(f"Type of documents: {type(documents)}")
len(documents)
documents

100%|██████████| 4/4 [00:00<00:00, 6884.37it/s]

Type of documents: <class 'list'>


[Document(metadata={'source': 'data/text_files/python.txt'}, page_content='## Python Programming: A Comprehensive Overview\n\nPython is one of the most popular, versatile, and beginner-friendly programming languages in the world. It was created by Guido van Rossum and emphasizes code readability, allowing programmers to express complex concepts in fewer \nlines of code than languages like C++ or Java.\n\nIn simple terms, Python provides a powerful, yet accessible, way to write software using a syntax that closely resembles plain English.\n\n***\n\n### Part 1: What Makes Python Unique? (The Philosophy)\n\nThe core philosophy behind Python is **simplicity and readability.**\n\n#### 1. High-Level Language\nPython is classified as a high-level language. This means that it is designed to be close to human language and abstracts away complex, low-level details (such as managing computer memory or dealing with machine registers). \nProgrammers can therefore focus on *what* the program needs t

## 3. Read PDFs using Langchain PdfLoader

Similar to loading text files, we can also load pdf files, we use PyPDFLoader or PyMuPDFLoader for this. Here we define the loader object.

In [12]:
loader = PyPDFLoader ("./data/pdf_files/paper1.pdf")
print(f"Type of loader: {type(loader)}")

Type of loader: <class 'langchain_community.document_loaders.pdf.PyPDFLoader'>


On calling the load() method of the loader object, we get back a non-primitive list which contains the contents of the documents. Note that each individual page is treated as a document.

In [13]:
pdf_document = loader.load()
print(f"Number of documents: {len(pdf_document)}. Note that this can correspond to each page in a document")
print(f"Type of pdf_document: {type(pdf_document)}")
pdf_document

Number of documents: 14. Note that this can correspond to each page in a document
Type of pdf_document: <class 'list'>


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-01T00:13:02+00:00', 'author': '', 'keywords': '', 'moddate': '2025-04-01T00:13:02+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/pdf_files/paper1.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='Merging Feed-Forward Sublayers for Compressed Transformers\nNeha Verma1 Kenton Murray1,2 Kevin Duh1,2\n1Center for Language and Speech Processing\n2Human Language Technology Center of Excellence\nJohns Hopkins University\n{nverma7, kenton}@jhu.edu, kevinduh@cs.jhu.edu\nAbstract\nWith the ubiquity of large deep learning mod-\nels and their growing number of use cases,\nthe need for high-quality compression tech-\nniques is growing in order to deploy these\nmodels widely across diverse hardware and\nmemory settings. In this work, we p

Here we define the object of the DirectoryLoader and provide it the directory and loader class which will specify the loader to be used on the individual files of that directory when calling the load() method.

In [14]:
dir_loader = DirectoryLoader(   "./data/pdf_files", 
                                glob="**/*.pdf",
                                loader_cls=PyPDFLoader,
                                show_progress=True
                            )
print(f"Type of dir_loader: {type(dir_loader)}")
dir_loader

Type of dir_loader: <class 'langchain_community.document_loaders.directory.DirectoryLoader'>


The load() method of the DirectoryLoader object is being called which will return a non-primitive list of all pages in the documents contained in the directory of our choice.

In [15]:
pdf_documents = dir_loader.load()
print(f"Type of pdf_documents: {type(pdf_documents)} \nNumber of Documents:{len(pdf_documents)}")
pdf_documents

100%|██████████| 5/5 [00:02<00:00,  2.37it/s]

Type of pdf_documents: <class 'list'> 
Number of Documents:58


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-01T00:13:02+00:00', 'author': '', 'keywords': '', 'moddate': '2025-04-01T00:13:02+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/pdf_files/paper1.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='Merging Feed-Forward Sublayers for Compressed Transformers\nNeha Verma1 Kenton Murray1,2 Kevin Duh1,2\n1Center for Language and Speech Processing\n2Human Language Technology Center of Excellence\nJohns Hopkins University\n{nverma7, kenton}@jhu.edu, kevinduh@cs.jhu.edu\nAbstract\nWith the ubiquity of large deep learning mod-\nels and their growing number of use cases,\nthe need for high-quality compression tech-\nniques is growing in order to deploy these\nmodels widely across diverse hardware and\nmemory settings. In this work, we pre

## 4. Embedding

In [16]:
# import sys
# import numpy as np
# import sklearn
# from sklearn.metrics.pairwise import cosine_similarity
# from sentence_transformers import SentenceTransformer
# import chromadb
# import uuid
# from typing import List, Dict, Any, Tuple


RAG queries documents by their embeddings in a fuzzy way instead of some form of a discrete key.

Embeddings are basically the weights of the second last layer of a text classifier. 
It contains almost all information of the text fed to the text classifier, but in a fixed size and format.
As embeddings are essentially lists of floating point numbers, they can be used to classify and distinguish one sentence from other.

Embeddings can also be conceptualised as n-dimentional representation of the sentence, where each dimention may vaguely represent a concept. 
Two sentences can be similar if they have close values at same indices, representing they are conceptually alike.

A query in RAG is also passed throught this truncated text classifier, which generates it's embeddings. 
The embedding of the query is then matched with the embeddings of all the stored documents.
The document or chunk which has the highest similarity with respect to the embedding is considered to be most closely related to the query.


Here we define the Embedding_Manager class which is responsible for generation of the embeddings.


In [ ]:
class Embedding_Manager:
    # To handle document embedding generation
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
            Initialize embedding manager
            model_name: Huggingface model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """
            Loads the selected model. 
            At times self.model cannot be set due to incorrect model-name or other issues.
        """
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimention {self.get_embedding_dimension()}")
        except Exception as e:
            print(f"Failed loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
            Generates embeddings for a list of texts

            Arguments:
                texts: List of text strings to embed
            
            Returns embeddings for each document/chunk in form of an array having shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts.")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    def get_embedding_dimension(self) -> int:
        """
            Returns the embedding dimension of the model.
            This function is just for informatics purpose and is used while loading the model or for later interrogation.

            In case the model is not loaded, it returns an ValueError
        """
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()

In [18]:
embedding_manager = Embedding_Manager()
print(f"Type of embedding_manager: {type(embedding_manager)}")

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5280.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimention 384
Type of embedding_manager: <class '__main__.Embedding_Manager'>


## 5. Chunking function

LLMs have a limited context window. 
Any sentence exceeding the limit of the context window will be rejected from processing by the LLM. 
So the text is divided into chunks of managable size, ideally just a bit less than the context window limit.

There are various text splitters provided by LangChain library which can be employed for chunking.
These text splitters fall under the following categories:
    1. Core text splitters:
    2. Token based splitters:
    3. Structure aware splitters:
    4. Code specific splitters:
    5. Document specific splitters:
    6. Semantic splitters: 
    7. Specialised splitters:
    8. Experimental splitters:

Some of them are:
    1. Character Text Splitter: splits text by using specified delimiters. The used can set them to be '\n', '\n\n', '\t ', ' ', etc., or any other character.
    2. Recursive Character Text Splitter: 


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def recursive_split_documents(documents,chunk_size=2000,chunk_overlap=200):
    """
    Splits documents into smaller chunks.
    
    Arguments:
        chunk_size: Maximum characters per chunk. This should be equal to the model's context window.
        chunk_overlap: Characters to overlap between chunks for preserving context.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,                              # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap,                        # 200 chars overlap for context
        length_function=len,                                # How to measure length
        separators=["\n\n", "\n", " ", ""]                  # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [20]:
chunks = recursive_split_documents(pdf_documents,chunk_size=1000 ,chunk_overlap=200)
print(f"Type of chunks: {type(chunks)}. \nLength of chunks: {len(chunks)}")
chunks

Split 58 documents into 362 chunks

Example chunk:
Content: Merging Feed-Forward Sublayers for Compressed Transformers
Neha Verma1 Kenton Murray1,2 Kevin Duh1,2
1Center for Language and Speech Processing
2Human Language Technology Center of Excellence
Johns Ho...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-01T00:13:02+00:00', 'author': '', 'keywords': '', 'moddate': '2025-04-01T00:13:02+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/pdf_files/paper1.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}
Type of chunks: <class 'list'>. 
Length of chunks: 362


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-01T00:13:02+00:00', 'author': '', 'keywords': '', 'moddate': '2025-04-01T00:13:02+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/pdf_files/paper1.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='Merging Feed-Forward Sublayers for Compressed Transformers\nNeha Verma1 Kenton Murray1,2 Kevin Duh1,2\n1Center for Language and Speech Processing\n2Human Language Technology Center of Excellence\nJohns Hopkins University\n{nverma7, kenton}@jhu.edu, kevinduh@cs.jhu.edu\nAbstract\nWith the ubiquity of large deep learning mod-\nels and their growing number of use cases,\nthe need for high-quality compression tech-\nniques is growing in order to deploy these\nmodels widely across diverse hardware and\nmemory settings. In this work, we pre

In [21]:
len(pdf_documents)

58

## 6. Vector Store

In [22]:
class VectorStore:
    """
        Manages document embeddings in a ChromaDB vector store
    """
    def __init__(self, collection_name: str = "chunks", persist_directory: str = "./data/vector_store"):
        """
            Initialize the vector store
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """
            Initialize ChromaDB client and collection
        """
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection  (   name=self.collection_name, 
                                                                        metadata={"description":"PDF document embeddings for RAG"}
                                                                    )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
            Add documents and their embeddings to vector store
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents not equal to number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store.")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())
            
            # Metadata and content can be used for filtering and retrieval later
            metadatas.append(metadata)
        
        # Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise

In [23]:
vector_store = VectorStore()
print(f"Type of vector_store: {type(vector_store)}")

Vector store initialized. Collection: chunks
Existing documents in collection: 0
Type of vector_store: <class '__main__.VectorStore'>


## 7. Extracting all texts from the chunks and creating embeddings

In [24]:
"""
    Convert the text to embeddings
"""
texts = [doc.page_content for doc in chunks]
len(texts)

362

In [25]:
"""
    Generate embeddings for the document chunks
"""
embeddings = embedding_manager.generate_embeddings(texts)
len(embeddings)

Generating embeddings for 362 texts.


Batches: 100%|██████████| 12/12 [00:00<00:00, 18.22it/s]

Generated embeddings with shape: (362, 384)


362

In [26]:
"""
    Store in vector store
"""
vector_store.add_documents(chunks,embeddings)

Adding 362 documents to vector store.
Successfully added 362 documents to vector store
Total documents in collection: 362


## 8.Retrieval from Vector Store

In [27]:
class RAGRetriever:
    """
        Handle query-based retrieval from vector store
    """
    def __init__(self, vector_store: VectorStore, embedding_manager: Embedding_Manager):
        """
            Initialize the RAG retriever
            vector_store: Vector store instance
            embedding_manager: Embedding manager instance
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Tuple[str, Dict[str, Any]]]:
        """
            Retrieve relevant documents for a query
            query: User query string
            top_k: Number of top results to return
            score_threshold: Minimum similarity score for retrieval
            returns: List of tuples (document content, metadata)
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance, so similarity = 1 - distance)
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents above the similarity threshold.")
            else:
                print(f"No documents retrieved for the query.")
                
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
    

In [28]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)
print(f"Type of rag_retriever: {type(rag_retriever)}")

Type of rag_retriever: <class '__main__.RAGRetriever'>


In [29]:
rag_retriever.retrieve("Event Prediction", top_k=3, score_threshold=0.01)

Retrieving documents for query: 'Event Prediction'
Top K: 3, Score threshold: 0.01
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the similarity threshold.


[{'id': 'doc_5e0d318b_162',
  'content': 'achieved through a dense transformation layer followed by\na softmax function. This process generates the predicted\nprobability distribution of the event type. We calculate the\ncross-entropy loss Lp using the true event type labels yi\nand the predicted probability distribution of event type Pi\n, i.e., Lp = − P\ni yi log(Pi), thus optimizing the model for\naccurate event type prediction.\nEvent time prediction For predicting event time, we add\nanother dense layer on top ofHL to learn the distribution pa-\nrameters of the temporal point process, specifically the scale\nparameter λ and shape parameter γ. The proposed frame-\nwork can use the Weibull distribution (Rinne 2008) to model\nthe intensity function. The exponential distribution, a spe-\ncific case of the Weibull distribution with γ = 1, has a con-\nstant intensity function suggesting events occur with a uni-\nform likelihood, irrespective of past occurrences. This char-\nacteristic m

In [30]:
rag_retriever.retrieve("what is feature based cycle aware time positional encoding", top_k=3, score_threshold=0.01)

Retrieving documents for query: 'what is feature based cycle aware time positional encoding'
Top K: 3, Score threshold: 0.01
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 221.16it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the similarity threshold.


[{'id': 'doc_f66df619_147',
  'content': 'to capture the relative temporal order of events in TPPs. Ex-\nisting methods can be classified into fixed (Vaswani et al.\n2017) and learned encoding (Kazemi et al. 2019; Xu et al.\n2020; Zhang et al. 2020; Xu et al. 2019; Li et al. 2021;\nDikeoulias, Amin, and Neumann 2022; Shaw, Uszkoreit,\nand Vaswani 2018; Raffel et al. 2020), but they fail to learn\nevent cycles based on event features. Research highlights the\nimportance of incorporating semantic features to accurately\nrepresent periodic patterns in real-world phenomena (Ke,\nHe, and Liu 2021; Zhang, Lee, and Lee 2019). To effectively\ncapture complex cyclic patterns in irregular time sequences,\nwe introduce a novel Feature-based Cycle-aware Time Po-\nsitional Encoding (FCPE), which integrates these essential\nsemantic aspects into the encoding of time intervals between\nevents.\nFormally, time positional encoding can be described as a\nfunction P : T → Rd×1, mapping the time domain T 